# CodeAlpha Data Analytics Internship
## Task 2: Exploratory Data Analysis (EDA) — Titanic Dataset

- **Intern**: Om Katkar
- **Dataset**: Titanic Passenger Survival (`dataset/titanic.csv`)
- **Libraries**: `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy.stats`

---
### Pre-Analysis Questions
1. **Demographic Disparity**: Did female passengers have a significantly higher survival rate than males?
2. **Socioeconomic Stratification**: Did passengers in 1st class experience significantly better survival rates compared to 2nd and 3rd class?
3. **Age & Vulnerability**: Were children prioritized for rescue compared to adults?
4. **Ticket Economics & Anomalies**: What is the distribution of ticket fares, and are there severe outliers?
5. **Hypothesis Testing**: Are the observed disparities in gender, class, and fare statistically significant under rigorous hypothesis tests?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded successfully!")

### 1. Data Ingestion & Structure Inspection

In [ ]:
df = pd.read_csv("dataset/titanic.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())

In [ ]:
# Missing values breakdown
null_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %	": (df.isnull().mean() * 100).round(2),
    "Dtype": df.dtypes
})
display(null_summary)

### 2. Anomaly & Outlier Detection (Fare)
Using the Interquartile Range (IQR) technique to detect extreme fare values.

In [ ]:
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper_cutoff = q3 + 1.5 * iqr
outliers = df[df["fare"] > upper_cutoff]
print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f} | Upper Cutoff: {upper_cutoff:.2f}")
print(f"Detected {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}% of dataset)")

### 3. Statistical Hypothesis Testing
Testing whether differences are statistically significant.

In [ ]:
# H1: Gender vs Survival (Chi-square test)
chi2_sex, p_sex, _, _ = stats.chi2_contingency(pd.crosstab(df["sex"], df["survived"]))
print(f"H1 (Sex vs Survival) Chi2: {chi2_sex:.4f}, p-value: {p_sex:.4e}")

# H2: Class vs Survival (Chi-square test)
chi2_class, p_class, _, _ = stats.chi2_contingency(pd.crosstab(df["pclass"], df["survived"]))
print(f"H2 (Class vs Survival) Chi2: {chi2_class:.4f}, p-value: {p_class:.4e}")

# H3: Fare differences (Two-sample T-test)
surv_fares = df[df["survived"] == 1]["fare"].dropna()
vict_fares = df[df["survived"] == 0]["fare"].dropna()
t_stat, p_t = stats.ttest_ind(surv_fares, vict_fares, equal_var=False)
print(f"H3 (Fare Difference) T-stat: {t_stat:.4f}, p-value: {p_t:.4e}")

### 4. Visualizations & Pattern Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df.groupby("sex")["survived"].mean().reset_index(), x="sex", y="survived", hue="sex", legend=False, ax=axes[0], palette=["#2b5c8f", "#d95f02"])
axes[0].set_title("Survival Rate by Gender")

sns.barplot(data=df.groupby(["pclass", "sex"])["survived"].mean().reset_index(), x="pclass", y="survived", hue="sex", ax=axes[1], palette=["#2b5c8f", "#d95f02"])
axes[1].set_title("Survival Rate by Class & Gender")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x="age", hue="survived", common_norm=False, fill=True, alpha=0.4, palette=["#c44e52", "#4c72b0"])
plt.title("Age Distribution: Survivors (1) vs Non-Survivors (0)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()